# NUMPY MASTERY — CHAPTERS 2–5

```
NUMPY MASTERY
Current Chapters: 2 (Creating ndarrays), 3 (ndarray Anatomy),
                  4 (Dtype Mastery), 5 (Indexing)
Progress:         5 / 75 chapters
Completed:        Ch1 - NumPy Foundation
Remaining:        70 chapters
```

This notebook covers four chapters together since they build on each other tightly:
how you **create** an array determines its **anatomy** (shape/dtype), and both determine
how **indexing** behaves.


In [ ]:
import numpy as np
import sys
print("NumPy version:", np.__version__)


---
# CHAPTER 2 — Creating ndarrays

## Concept
NumPy gives you two families of creation functions:
1. **From existing data**: `np.array()`, `np.asarray()`
2. **From a shape/pattern, no source data needed**: `np.zeros()`, `np.ones()`, `np.empty()`,
   `np.full()`, `np.arange()`, `np.linspace()`, `np.logspace()`, `np.eye()`, `np.identity()`,
   `np.diag()`, `np.fromfunction()`, `np.fromiter()`

## Why it matters
In real pipelines you rarely hand-type array contents. You either convert data you already
have (`asarray`), or you allocate a buffer of a known shape to fill in later (`zeros`/`empty`),
or you generate a numeric sequence for indexing/plotting/simulation (`arange`/`linspace`).
Picking the *wrong* one silently wastes memory or introduces bugs (e.g., `empty()` contains
garbage, not zeros).

## Industry use case
Initializing neural network weight matrices (`np.zeros`, or random init in Ch.20), building a
time axis for a time-series plot (`linspace`), pre-allocating an output buffer for a batch
job (`empty`, filled in-place for speed), building a one-hot / identity matrix for categorical
encoding (`eye`).


In [ ]:
# --- np.array() vs np.asarray() ---
original = np.array([1, 2, 3])

# np.array() ALWAYS copies by default
copy_via_array = np.array(original)
copy_via_array[0] = 999
print("np.array()   copies -> original unaffected:", original)

# np.asarray() does NOT copy if input is already a compatible ndarray
view_via_asarray = np.asarray(original)
view_via_asarray[0] = 999
print("np.asarray() reuses -> original IS affected:", original)


### Expected Output
```
np.array()   copies -> original unaffected: [1 2 3]
np.asarray() reuses -> original IS affected: [999   2   3]
```

### `np.array()` vs `np.asarray()` — the real difference
- `np.array(data, copy=True)` by default — **always** makes a new array, even if `data` is
  already an ndarray of the right dtype.
- `np.asarray(data)` — returns the **input unchanged** if it's already an ndarray with a
  compatible dtype; only copies when necessary (e.g., converting a list, or changing dtype).

**Rule of thumb:** use `asarray()` inside library/function code where you don't want to pay for
an unnecessary copy of data the caller already gave you as an array. Use `array()` when you
specifically need an independent copy (e.g., you're about to mutate it and don't want to affect
the caller's data).

### Common Mistake
Using `np.array()` inside a hot loop or a function called millions of times, when the input is
already guaranteed to be an ndarray — this silently doubles memory traffic for no benefit.

### Interview Insight
> "When would you use `asarray` over `array`?"
Good answer: "Inside a function that shouldn't unnecessarily copy caller-owned array data —
`asarray` avoids the copy if the input is already array-like with a compatible dtype; `array`
guarantees an independent copy, which matters when you're about to mutate the result."


In [ ]:
# --- Shape-based creation functions ---
zeros = np.zeros((2, 3))          # filled with 0.0, dtype float64 by default
ones  = np.ones((2, 3))           # filled with 1.0
empty = np.empty((2, 3))          # UNINITIALIZED garbage values - fast, dangerous
full  = np.full((2, 3), 7)        # filled with a constant

print("zeros:\n", zeros)
print("ones:\n", ones)
print("empty (garbage!):\n", empty)
print("full:\n", full)


### `zeros()` vs `empty()`
`np.empty()` skips the zero-initialization step, so it's *marginally* faster to allocate — but
the contents are **whatever bytes were already in that memory**, not zeros. Use it only when you
are 100% certain every element will be overwritten before being read (e.g., you immediately fill
it in a loop). Using `empty()` and then reading unwritten elements is a classic silent-bug source
(NaNs, huge numbers, occasionally zeros — non-deterministic).

### Common Mistake
Using `np.empty()` "for performance" and then reading a value before writing to it — the bug
may not reproduce consistently, making it very hard to debug in CI.


In [ ]:
# --- Sequence generators ---
a1 = np.arange(0, 10, 2)          # start, stop (exclusive), step -> like Python range
a2 = np.linspace(0, 1, 5)         # start, stop (INCLUSIVE), num points -> evenly spaced
a3 = np.logspace(0, 3, 4)         # 10**0 ... 10**3, 4 points, log-spaced

print("arange  :", a1)
print("linspace:", a2)
print("logspace:", a3)

# --- Matrices ---
eye = np.eye(3)                   # identity matrix
diag = np.diag([1, 2, 3])         # diagonal matrix from a vector
extract_diag = np.diag(eye * 5)   # extract diagonal from a matrix

print("eye:\n", eye)
print("diag:\n", diag)
print("extracted diagonal:", extract_diag)


### `arange()` vs `linspace()`
| | `arange(start, stop, step)` | `linspace(start, stop, num)` |
|---|---|---|
| You specify | the **step size** | the **number of points** |
| `stop` | exclusive | **inclusive** by default |
| Float safety | can suffer floating-point drift over many steps | numerically stable, exact endpoint count |

**Rule of thumb:** use `linspace` whenever you need an exact, predictable *number* of points
(e.g., 100 points for a plot x-axis) — `arange` with a float step can produce one extra or one
fewer element than expected due to floating-point rounding.

### Real-world use
- `linspace` → generating an x-axis for plotting a function, or a probability grid for
  simulation.
- `eye`/`identity` → initializing regularization terms (ridge regression: `X.T @ X + λ·I`).
- `diag` → constructing a covariance-like diagonal matrix, or extracting variances from a
  covariance matrix.

### Exercise
Predict the output length of `np.arange(0, 1, 0.1)` — is it 10 elements? Run it and explain any
surprise using what you know about floating-point representation (we'll cover this formally in
Chapter 27 — Numerical Stability).


---
# CHAPTER 3 — ndarray Anatomy

## Concept
Every `ndarray` is described by five things: **dtype** (element type), **shape** (size along
each axis), **ndim** (number of axes), **strides** (bytes to step to move one index along each
axis), and a pointer to a **contiguous data buffer**. Understanding *strides* is what separates
someone who "uses" NumPy from someone who understands it.

## Why it matters
`.T` (transpose), broadcasting, and many "free" operations (like transpose) work by changing
*metadata* (strides/shape), not by moving data. If you don't understand this, view/copy bugs
(Chapter 8) will confuse you for years.


In [ ]:
arr = np.arange(24).reshape(2, 3, 4)

print("arr.ndim    :", arr.ndim)      # 3 axes
print("arr.shape   :", arr.shape)     # (2, 3, 4)
print("arr.size    :", arr.size)      # 24 total elements (2*3*4)
print("arr.dtype   :", arr.dtype)     # int64
print("arr.itemsize:", arr.itemsize, "bytes per element")
print("arr.nbytes  :", arr.nbytes, "bytes total (= size * itemsize)")
print("arr.strides :", arr.strides, "bytes to step along each axis")
print("arr.flags['C_CONTIGUOUS']:", arr.flags['C_CONTIGUOUS'])


### Expected Output
```
arr.ndim    : 3
arr.shape   : (2, 3, 4)
arr.size    : 24
arr.dtype   : int64
arr.itemsize: 8 bytes per element
arr.nbytes  : 192 bytes total (= size * itemsize)
arr.strides : (96, 32, 8) bytes to step along each axis
arr.flags['C_CONTIGUOUS']: True
```

### Reading `strides` — the key mental model
`strides = (96, 32, 8)` for shape `(2, 3, 4)`, dtype `int64` (8 bytes/element) means:
- Move **8 bytes** to go to the next element along axis 2 (the fastest-changing axis — this is
  "C order" / row-major: last axis is contiguous).
- Move **32 bytes** (= 4 elements × 8 bytes) to go to the next "row" along axis 1.
- Move **96 bytes** (= 12 elements × 8 bytes) to go to the next "block" along axis 0.

This is *why* `arr.T` is free (no data copy): NumPy just reverses the shape and strides tuples,
and reads the same underlying buffer in a different order.

### Visualizing dimensions → real ML data
```
1D  (5,)          -> a single feature vector / 1 row of measurements
2D  (3, 4)        -> tabular data: 3 samples, 4 features  (classic ML feature matrix)
3D  (2, 3, 4)     -> e.g. 2 grayscale images, each 3x4 pixels  (batch, H, W)
4D  (2, 3, 4, 5)  -> e.g. 2 RGB-ish images (H=3, W=4, C=5)     (batch, H, W, channels)
```

### Common Mistake
Assuming `.T` or `.reshape()` always cost the same as copying data — they usually don't; they
manipulate `shape`/`strides` metadata only, when possible. This matters for performance
reasoning (Chapter 28–30) but also means mutating a transposed array mutates the original.

### Interview Insight
> "Why does transposing a large matrix seem instantaneous?"
Because `.T` doesn't move any data — it swaps the shape and strides metadata and returns a
*view* into the same buffer, read in a different stride pattern. Actual computation only
happens once you materialize/copy it or perform an operation that requires a particular memory
layout (e.g., some BLAS calls require contiguous memory and will silently copy internally).


---
# CHAPTER 4 — Data Types / dtype Mastery

## Concept
`dtype` fixes how many bytes represent each element and how those bytes are interpreted
(signed/unsigned integer, float, complex, bool). This single choice governs **precision**,
**range**, **memory**, and (on GPUs) **throughput**.

## Why it matters
In deep learning, switching model weights from `float32` to `float16`/`bfloat16` roughly halves
memory and can significantly speed up GPU training — at the cost of precision, which can matter
for numerical stability (Chapter 27, 45). Choosing `int8` vs `int64` for a huge categorical/ID
column can be the difference between a dataset fitting in RAM or not.


In [ ]:
for dt in [np.int8, np.int16, np.int32, np.int64, np.uint8,
           np.float16, np.float32, np.float64]:
    info = np.iinfo(dt) if np.issubdtype(dt, np.integer) else np.finfo(dt)
    print(f"{dt.__name__:10s} itemsize={np.dtype(dt).itemsize} bytes   range: [{info.min}, {info.max}]")


### Expected Output (abbreviated)
```
int8       itemsize=1 bytes   range: [-128, 127]
int16      itemsize=2 bytes   range: [-32768, 32767]
int32      itemsize=4 bytes   range: [-2147483648, 2147483647]
int64      itemsize=8 bytes   range: [-9223372036854775808, 9223372036854775807]
uint8      itemsize=1 bytes   range: [0, 255]
float16    itemsize=2 bytes   range: [-65500.0, 65500.0]
float32    itemsize=4 bytes   range: [-3.4028235e+38, 3.4028235e+38]
float64    itemsize=8 bytes   range: [-1.7976931348623157e+308, 1.7976931348623157e+308]
```


In [ ]:
# --- Integer overflow demo ---
small = np.array([120, 125, 130], dtype=np.int8)
print("int8 array:", small)
overflowed = small + np.int8(10)   # wraps around silently, no error!
print("small + 10 as int8 (WRAPS AROUND):", overflowed)

# --- Floating point precision demo ---
print()
print("0.1 + 0.2 == 0.3 ?", 0.1 + 0.2 == 0.3)
print("0.1 + 0.2 =", 0.1 + 0.2)
print("np.isclose(0.1 + 0.2, 0.3) ?", np.isclose(0.1 + 0.2, 0.3))


### Expected Output
```
int8 array: [120 125 130]
small + 10 as int8 (WRAPS AROUND): [ -126  -121  -116]
```
`130` can't even be represented in `int8` (max 127) — it silently wrapped to `-126` at creation
time. This is **not an error**, which is exactly why it's dangerous in production.

```
0.1 + 0.2 == 0.3 ? False
0.1 + 0.2 = 0.30000000000000004
np.isclose(0.1 + 0.2, 0.3) ? True
```

### Why this matters for ML
- **Silent overflow**: casting large label/ID columns to `int8`/`int16` "to save memory" without
  checking the actual value range will corrupt data silently — no exception is raised.
- **float32 vs float64 in deep learning**: `float32` is the DL default (half the memory/bandwidth
  of `float64`, sufficient precision for gradient descent). `float16`/`bfloat16` push further for
  GPU throughput but require care (loss scaling) to avoid vanishing gradients — this is *why*
  "mixed precision training" is a whole subfield.
- **Never compare floats with `==`.** Always use `np.isclose()` / `np.allclose()` (Chapter 27).

### `astype()`
```python
x = np.array([1.9, 2.5, -1.9])
x.astype(np.int32)   # -> [1, 2, -1]  (TRUNCATES toward zero, does not round!)
```

### Common Mistake
Assuming `.astype(int)` rounds — it **truncates** toward zero. Use `np.round(x).astype(int)` if
you want rounding.

### Interview Insight
> "Is float32 always better than float64?"
No — it's a memory/speed vs. precision trade-off. float32 is standard for deep learning (memory
+ GPU throughput matter more than the last few bits of precision, and models are trained with
techniques robust to small numeric error). float64 remains important for scientific computing,
financial calculations, and numerically sensitive linear algebra (e.g., ill-conditioned matrix
inversion) where accumulated rounding error can matter.


---
# CHAPTER 5 — Indexing

## Concept
NumPy indexing extends Python's list indexing/slicing to multiple dimensions using a single
comma-separated tuple: `arr[axis0_index, axis1_index, ...]`.

## Why it matters
Selecting rows, columns, or sub-blocks of a feature matrix, cropping regions of an image tensor,
or slicing a time window out of a time series — all of this is indexing.


In [ ]:
arr = np.arange(1, 13).reshape(3, 4)
print("arr:\n", arr)
print()

print("arr[0]      (first row)          :", arr[0])
print("arr[-1]     (last row)           :", arr[-1])
print("arr[0, 2]   (row 0, col 2)       :", arr[0, 2])
print("arr[1, :]   (whole row 1)        :", arr[1, :])
print("arr[:, 2]   (whole column 2)     :", arr[:, 2])
print("arr[0:2, 1:3] (sub-block)        :\n", arr[0:2, 1:3])
print("arr[::-1]   (reverse rows)       :\n", arr[::-1])
print("arr[:, ::2] (every 2nd column)   :\n", arr[:, ::2])


### Expected Output
```
arr:
 [[ 1  2  3  4]
  [ 5  6  7  8]
  [ 9 10 11 12]]

arr[0]      (first row)          : [1 2 3 4]
arr[-1]     (last row)           : [ 9 10 11 12]
arr[0, 2]   (row 0, col 2)       : 3
arr[1, :]   (whole row 1)        : [5 6 7 8]
arr[:, 2]   (whole column 2)     : [ 3  7 11]
arr[0:2, 1:3] (sub-block)        :
 [[2 3]
  [6 7]]
arr[::-1]   (reverse rows)       :
 [[ 9 10 11 12]
  [ 5  6  7  8]
  [ 1  2  3  4]]
arr[:, ::2] (every 2nd column)   :
 [[ 1  3]
  [ 5  7]
  [ 9 11]]
```

### Shape Analysis rule
- Indexing with a **single integer** on an axis **removes** that axis from the result
  (`arr[0]` on a `(3,4)` array → shape `(4,)`, not `(1,4)`).
- Indexing with a **slice** (even `0:1`) **keeps** that axis
  (`arr[0:1]` → shape `(1,4)`).

This distinction ("does this axis survive?") is one of the most common sources of shape bugs in
real pipelines — predict it before running, always.

### Important preview: Slicing returns a VIEW, not a copy
```python
row = arr[0, :]
row[0] = 999   # THIS MUTATES THE ORIGINAL arr TOO
```
We dedicate all of **Chapter 8** to this because it is the single most common source of "spooky
action at a distance" bugs in NumPy code — a function that "just reads a slice" can silently
corrupt the caller's original array. For now: know that basic slicing (using `:`) shares memory;
we'll cover fancy/boolean indexing's different behavior in Chapters 6–8.

### Interview Insight
> "What shape does `arr[1:2, :]` have on a `(3, 4)` array, versus `arr[1, :]`?"
`arr[1:2, :]` → `(1, 4)` (slice keeps the axis). `arr[1, :]` → `(4,)` (integer index drops the
axis). Being able to answer this instantly signals real fluency, not memorized syntax.


---
## Chapters 2–5 Summary

- **Creation**: `array()` copies, `asarray()` avoids unnecessary copies; `zeros`/`ones`/`full`
  give predictable content, `empty()` is fast but uninitialized (dangerous); `arange` is
  step-based, `linspace` is count-based and float-safe.
- **Anatomy**: `shape`, `ndim`, `size`, `dtype`, `itemsize`, `nbytes`, `strides` fully describe
  an array. Strides explain why transpose/reshape are often "free" (metadata-only).
- **dtype**: fixes precision, range, and memory. Integer overflow and float rounding are silent
  — never compare floats with `==`; watch dtype ranges when downcasting.
- **Indexing**: integer index drops an axis, slice keeps it. Basic slices are views (share
  memory) — a preview of Chapter 8's deep dive.

---

## Mastery Exercises

**Exercise 1 (Prediction):** Given `x = np.ones((4, 5), dtype=np.float32)`, predict `x.nbytes`,
`x.itemsize`, and `x.shape` after `x.T`. Then verify.

**Exercise 2:** Create a `(5,)` array of 5 evenly spaced points between -1 and 1 **inclusive**,
using the function best suited for "I want an exact count of points."

**Coding Challenge:** Write a function `safe_downcast(arr, dtype)` that converts a NumPy array to
a smaller integer dtype **only if** no values would overflow, otherwise raises a `ValueError`
with a clear message. (Hint: use `np.iinfo`.) This is exactly the kind of defensive check
production data-loading code needs.

**Debugging Challenge:**
```python
def get_first_row(matrix):
    return matrix[0, :]

data = np.array([[1, 2, 3], [4, 5, 6]])
row = get_first_row(data)
row[:] = 0
print(data)   # a teammate is surprised this is NOT [[1,2,3],[4,5,6]] anymore. Why?
```
Explain the root cause (you'll formalize the fix pattern in Chapter 8).

**Interview Questions:**
1. Difference between `np.array()` and `np.asarray()`, and when each is appropriate?
2. What are strides, and why does understanding them explain why `.T` is "free"?
3. Why does `astype(int)` sometimes surprise people coming from other languages/rounding
   conventions?
4. What shape does `arr[2]` return vs `arr[2:3]` on a `(10, 6)` array?

**Quiz:**
- Q1: True/False — `np.empty((3,3))` is guaranteed to contain zeros.
- Q2: Which creation function should you use when you need *exactly* 50 points between two
  values, endpoints included?
- Q3: If `arr.strides == (40, 8)` and `dtype` is `float64` (8 bytes), how many elements are in
  each row?

---


